# 03 - Anomaly detection + classification  (Member 3)

- IsolationForest: inspect the 47 anomalies found on the training data
- Compare: majority class vs DecisionTree(depth 3) vs RandomForest
- Feature importance, ROC curve, confusion matrix
- Ablation: 10 raw features vs 13 (with cluster + PCA columns)

**Rules:** fit every scaler/PCA/model on `Xtr` ONLY (never on the full `df`). Evaluate on `Xte` at the end. Edit only this notebook, on your own branch.

**Deliverables:** (1) notebook runs top-to-bottom with no errors, (2) 2-3 high-quality PNG plots for the slides, (3) a 5-line summary answering: *why did the result come out like this?*

> **Important:** `Overtraining_Risk_Label` is a RULE built from the input features (see `build_rule_label`), so high scores = rule re-discovery, not real injury prediction. Also: 195 test rows -> a 0.004 F1 gap is about one sample. Use 5-fold CV before claiming that one variant is better.

In [ ]:
import sys; sys.path.insert(0, "..")          # project root, so `backend` can be imported
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from backend.ml_pipeline import load_data_and_split, SEED
from backend.config import FEATURE_COLS

# The ONE official split (seed 42, stratified). Do NOT make your own split.
df, Xtr, Xte, yc_tr, yc_te, yr_tr, yr_te = load_data_and_split()
print(Xtr.shape, Xte.shape)                   # expected: (778, 10) (195, 10)

In [ ]:
from backend.ml_pipeline import build_rule_label
print('rule reproduces the label on all rows:', (build_rule_label(df) == df['Overtraining_Risk_Label']).all())

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
models = {
  'majority': DummyClassifier(strategy='most_frequent'),
  'tree depth 3': DecisionTreeClassifier(max_depth=3, random_state=SEED),
  'random forest': RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced', random_state=SEED),
}
for name, m in models.items():
    m.fit(Xtr, yc_tr); p = m.predict(Xte)
    print(name, 'F1', round(f1_score(yc_te, p), 4))   # reference: 0.7888 / 0.9127 / ~0.944 (10 raw features)

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(Xtr)
iso = IsolationForest(contamination=0.06, random_state=SEED).fit(scaler.transform(Xtr))
anom = Xtr[iso.predict(scaler.transform(Xtr)) == -1]
print(len(anom), 'anomalies (reference: 47)')
anom.describe().T[['mean', 'min', 'max']]

**TODO:** feature importance bar chart - ROC curve - confusion matrix - ablation 10 vs 13 features WITH `cross_val_score` (5-fold) - what do the 47 anomalies have in common?